# Wan2.1-T2V-1.3B — Video Generation on AMD ROCm

**Platform:** AMD Radeon Cloud (`vllm_omni:20260609`)  
**Model:** [`Wan-AI/Wan2.1-T2V-1.3B`](https://modelscope.cn/models/Wan-AI/Wan2.1-T2V-1.3B)  
**Architecture:** Diffusion Transformer — 1.3B parameters  
**License:** Apache 2.0

Wan2.1 is an open-source video generation model from Alibaba. The 1.3B variant fits in ~8 GB VRAM and runs comfortably on this 48 GB gfx1100.  
Weights are downloaded from **ModelScope** (no HuggingFace access needed).

---

## Hardware

| Model | VRAM (BF16) | This machine |
|---|---|---|
| **Wan2.1-T2V-1.3B** | **~8 GB** | **✅ gfx1100 (48 GB)** |
| Wan2.1-T2V-14B | ~40 GB | ✅ fits |

## 0. Environment Verification

Verify that the ROCm stack is active and GPUs are visible. On AMD hardware, PyTorch exposes the HIP backend through the standard `torch.cuda` API — `torch.cuda.is_available()` returns `True` and `torch.version.hip` contains the ROCm version.

In [1]:
import subprocess, sys, os

# --- ROCm / GPU info ---
print("=" * 55)
print("ROCm & GPU Environment")
print("=" * 55)

result = subprocess.run(["rocm-smi", "--showproductname", "--showmeminfo", "vram"],
                        capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "rocm-smi not found — skipping")

import torch
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA (HIP) avail : {torch.cuda.is_available()}")
print(f"ROCm/HIP version : {getattr(torch.version, 'hip', 'N/A')}")
print(f"GPU count        : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    vram_gb = props.total_memory / (1024**3)
    print(f"  GPU {i}: {props.name}  |  VRAM: {vram_gb:.1f} GB")

total_vram = sum(
    torch.cuda.get_device_properties(i).total_memory
    for i in range(torch.cuda.device_count())
) / (1024**3)
print(f"\nTotal VRAM across all GPUs: {total_vram:.1f} GB")

COSMOS_NANO_REQ_GB = 32
if total_vram >= COSMOS_NANO_REQ_GB:
    print(f"✅ Sufficient VRAM for Cosmos3-Nano ({COSMOS_NANO_REQ_GB} GB required)")
else:
    print(f"⚠️  Total VRAM ({total_vram:.0f} GB) may be insufficient for Cosmos3-Nano.")
    print("   Cosmos3-Nano requires ~32 GB VRAM (BF16).")

ROCm & GPU Environment


============================ ROCm System Management Interface ============================
================================== Memory Usage (Bytes) ==================================
GPU[0]		: VRAM Total Memory (B): 51522830336
GPU[0]		: VRAM Total Used Memory (B): 28012544
====================================== Product Info ======================================
GPU[0]		: Card Series: 		AMD Radeon Graphics
GPU[0]		: Card Model: 		0x744b
GPU[0]		: Card Vendor: 		Advanced Micro Devices, Inc. [AMD/ATI]
GPU[0]		: Card SKU: 		D7070910
GPU[0]		: Subsystem ID: 	0x0e0c
GPU[0]		: Device Rev: 		0x00
GPU[0]		: Node ID: 		2
GPU[0]		: GUID: 		6853
GPU[0]		: GFX Version: 		gfx1100
================================== End of ROCm SMI Log ===================================

PyTorch version  : 2.9.1+gitff65f5b
CUDA (HIP) avail : True
ROCm/HIP version : 7.2.53211-e1a6bc5663
GPU count        : 1
  GPU 0: AMD Radeon Graphics  |  VRAM: 48.0 GB

Total VRAM across all GPUs: 48.0 GB
✅ 

## 1. Install Dependencies

The `vllm_omni` container ships with PyTorch ROCm and vLLM-Omni. We additionally need:
- **`diffusers` from git** — `Cosmos3OmniPipeline` is not on PyPI yet (as of June 2026)
- **`cosmos_guardrail`** — NVIDIA's content safety filter (optional but recommended)
- **`huggingface_hub`** — for downloading model assets and example prompts

In [ ]:
import subprocess, sys

def pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "--trusted-host", "pypi.org",
           "--trusted-host", "files.pythonhosted.org",
           *args]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"pip warning: {result.stderr[-500:]}")
    return result.returncode == 0

pip("--upgrade", "huggingface_hub")
pip("modelscope", "accelerate", "transformers", "sentencepiece",
    "imageio", "imageio-ffmpeg", "av", "Pillow", "ftfy")

# diffusers needs to be recent enough to have WanPipeline
try:
    from diffusers import WanPipeline
    print("WanPipeline already available.")
except ImportError:
    pip("--upgrade", "diffusers")
    print("diffusers upgraded. Restart kernel if WanPipeline still missing.")

print("✅ Dependencies ready.")

## 2. Download Model Weights from ModelScope

Wan2.1-T2V-1.3B weights are hosted on ModelScope (Chinese model hub). The `modelscope` SDK downloads them without needing HuggingFace access.  
Total download size: **~2.7 GB**.

In [ ]:
import os
from modelscope import snapshot_download

# Download from ModelScope — no HuggingFace access required
# Wan2.1-T2V-1.3B is ~2.7 GB in BF16 safetensors
MODEL_LOCAL_DIR = "/workspace/Wan2.1-T2V-1.3B"

if os.path.exists(os.path.join(MODEL_LOCAL_DIR, "config.json")):
    print(f"Model already downloaded at {MODEL_LOCAL_DIR}")
else:
    print("Downloading Wan2.1-T2V-1.3B from ModelScope (~2.7 GB)...")
    MODEL_LOCAL_DIR = snapshot_download(
        "Wan-AI/Wan2.1-T2V-1.3B",
        cache_dir="/workspace/Wan2.1-T2V-1.3B",
    )
    print(f"Downloaded to: {MODEL_LOCAL_DIR}")

print(f"Model path: {MODEL_LOCAL_DIR}")

## 3. ROCm Environment Variables

A few environment variables improve performance and compatibility on AMD GPUs.

In [4]:
import os, subprocess

# Disable NUMA balancing (recommended for ROCm MI300X performance)
subprocess.run(
    ["sudo", "sh", "-c", "echo 0 > /proc/sys/kernel/numa_balancing"],
    capture_output=True,
)

# Enable AMD's Triton-based Flash Attention kernel (MI200/MI300 CDNA GPUs)
os.environ["FLASH_ATTENTION_TRITON_AMD_ENABLE"] = "TRUE"

# Use better error messages for HIP kernel failures during debugging
os.environ["TORCH_USE_HIP_DSA"] = "1"

# Ensure BF16 is used (the only tested precision for Cosmos3)
# FP16 can overflow on long video generation; FP32 doubles memory usage
os.environ["PYTORCH_NO_CUDA_MEMORY_CACHING"] = "0"  # keep caching on for performance

# Optional: restrict to specific GPUs (equivalent of CUDA_VISIBLE_DEVICES on ROCm)
# os.environ["ROCR_VISIBLE_DEVICES"] = "0,1,2,3"

import torch
# Verify BF16 support on this GPU
if torch.cuda.is_available():
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"BF16 supported on device 0: {bf16_ok}")
    if not bf16_ok:
        print("WARNING: BF16 not supported. Cosmos3 may not run correctly on this GPU.")

print("ROCm environment configured.")

BF16 supported on device 0: True
ROCm environment configured.


---

## 4. Generate Video

Wan2.1-T2V-1.3B generates short video clips from text prompts. Default output is 81 frames at 16fps (~5 seconds). Reduce `num_frames` to 25 for faster testing.

In [ ]:
import torch
from diffusers import WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler

print(f"GPU: {torch.cuda.get_device_properties(0).name}")
print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

pipe = WanPipeline.from_pretrained(
    MODEL_LOCAL_DIR,
    torch_dtype=torch.bfloat16,
)
pipe = pipe.to("cuda")

print("✅ Pipeline loaded.")
print(f"VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.1f} GB")

### 4a. Quick Test (25 frames)

Run this first to verify the pipeline works before committing to a full 81-frame generation.

In [ ]:
from diffusers.utils import export_to_video
from IPython.display import Video

# Quick test: 25 frames = ~1.5s, much faster than full generation
output = pipe(
    prompt="A cat walks on the grass, realistic style.",
    num_frames=25,
    height=480,
    width=832,
    num_inference_steps=20,
    guidance_scale=5.0,
    generator=torch.Generator(device="cuda").manual_seed(42),
)

out_path = "/tmp/wan2_quick.mp4"
export_to_video(output.frames[0], out_path, fps=16)
print(f"Saved: {out_path}")
Video(out_path, width=640)

### 5b. Text-to-Video

`num_frames=189` at 24fps produces a ~7.875s clip — the default generation length in the model card. Lower `num_inference_steps` (e.g. 20) reduces generation time at a small quality cost.

In [ ]:
from diffusers.utils import export_to_video
from IPython.display import Video

prompt = (
    "A cat walks on the grass, realistic style."
)

# 81 frames @ 16fps = ~5s clip; use 25 frames for a quick test
output = pipe(
    prompt=prompt,
    num_frames=81,
    height=480,
    width=832,
    num_inference_steps=50,
    guidance_scale=5.0,
    generator=torch.Generator(device="cuda").manual_seed(42),
)

out_path = "/tmp/wan2_t2v.mp4"
export_to_video(output.frames[0], out_path, fps=16)
print(f"Saved: {out_path}")
Video(out_path, width=640)

### 5c. Image-to-Video

Provide a conditioning image (PIL Image or local path) alongside the text prompt. The model generates a video that continues the visual story from the given frame.

In [ ]:
from PIL import Image
import requests
from io import BytesIO
from diffusers.utils import export_to_video
from IPython.display import Video, display

img_url = "https://huggingface.co/nvidia/Cosmos3-Nano/resolve/main/assets/example_i2v_input.jpg"
response = requests.get(img_url, timeout=30)
cond_image = Image.open(BytesIO(response.content)).convert("RGB")
print(f"Conditioning image size: {cond_image.size}")
display(cond_image)

prompt_i2v = (
    "Starting from this frame, a robotic arm begins to move with precision across the workspace. "
    "The gripper extends toward the nearest object, grasps it gently, and lifts it in a smooth arc. "
    "The background remains static. Photorealistic, steady camera, 4K quality."
)

result_i2v = pipe(
    prompt=prompt_i2v,
    image=cond_image,    # conditioning frame; triggers image-to-video mode
    num_frames=189,
    height=720,
    width=1280,
    num_inference_steps=35,
    guidance_scale=6.0,
    generator=torch.Generator(device="cuda").manual_seed(17),
)

out_path_i2v = "/tmp/cosmos3_nano_i2v.mp4"
export_to_video(result_i2v.video, out_path_i2v, fps=24)
print(f"Saved: {out_path_i2v}")
Video(out_path_i2v, width=640)

---

## Path B — vLLM-Omni Server (Production API)

This path uses the vLLM-Omni server already present in the `vllm_omni:20260609` container. It exposes an OpenAI-compatible REST API, making it suitable for multi-user serving and integration with existing inference infrastructure.

### 6. Start the vLLM-Omni Server

**ROCm-specific differences from the NVIDIA model card:**
- `--use-hsdp` and `--ulysses-degree` are NVLink-specific flags — omit them
- Use `--tensor-parallel-size` for multi-GPU distribution
- `HIP_VISIBLE_DEVICES` (or `ROCR_VISIBLE_DEVICES`) instead of `CUDA_VISIBLE_DEVICES`

In [ ]:
import subprocess, time, requests as req, os, torch

n_gpus = torch.cuda.device_count()
SERVER_PORT = 8000
SERVER_URL = f"http://localhost:{SERVER_PORT}"

# NOTE: If you already ran Path A, free GPU memory first:
# del pipe; import gc; gc.collect(); torch.cuda.empty_cache()

serve_cmd = [
    "vllm", "serve", MODEL_ID,
    "--omni",
    "--host", "0.0.0.0",
    "--port", str(SERVER_PORT),
    "--tensor-parallel-size", str(n_gpus),  # distribute across all GPUs
    "--dtype", "bfloat16",
    "--init-timeout", "1800",  # allow 30 min for 64B model load
]

print("Starting vLLM-Omni server...")
print("Command:", " ".join(serve_cmd))
print(f"Tensor parallel size: {n_gpus} GPU(s)")

server_proc = subprocess.Popen(
    serve_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Poll until server is ready (health endpoint becomes 200)
print("Waiting for server to be ready (may take 5-15 min for first load)...")
for attempt in range(120):
    time.sleep(10)
    try:
        r = req.get(f"{SERVER_URL}/health", timeout=5)
        if r.status_code == 200:
            print(f"\nServer ready after {(attempt+1)*10}s")
            break
    except Exception:
        pass
    if attempt % 3 == 0:
        # Show last line of server output
        line = server_proc.stdout.readline().strip()
        if line:
            print(f"  Server log: {line}")
else:
    print("Server did not become ready within 20 minutes. Check logs above.")

### 7. Text-to-Video via REST API

In [ ]:
import requests, json
from pathlib import Path
from IPython.display import Video

prompt_api = (
    "An autonomous vehicle navigates a rain-slicked urban intersection at night. "
    "Streetlights reflect off wet asphalt. Pedestrians with umbrellas cross at a crosswalk. "
    "The vehicle smoothly decelerates, yields, then accelerates through the intersection. "
    "Dashcam perspective, realistic, 4K."
)

negative_prompt = {
    "text": "blurry, low quality, artifact, distorted, flickering, watermark, text overlay"
}

data = {
    "prompt": prompt_api,
    "negative_prompt": json.dumps(negative_prompt),
    "size": "1280x720",
    "num_frames": "189",
    "fps": "24",
    "num_inference_steps": "35",
    "guidance_scale": "6.0",
    "flow_shift": "10.0",
    "seed": "42",
}

print("Sending text-to-video request...")
response = requests.post(
    f"{SERVER_URL}/v1/videos/sync",
    data=data,
    headers={"Accept": "video/mp4"},
    timeout=300,
)
response.raise_for_status()

out_path = Path("/tmp/cosmos3_nano_api_t2v.mp4")
out_path.write_bytes(response.content)
print(f"Saved: {out_path}  ({len(response.content) / 1024:.0f} KB)")
Video(str(out_path), width=640)

### 8. Image-to-Video via REST API

In [ ]:
import requests, json, mimetypes
from pathlib import Path
from io import BytesIO
from PIL import Image
from IPython.display import Video, display

img_url = "https://huggingface.co/nvidia/Cosmos3-Nano/resolve/main/assets/example_i2v_input.jpg"
img_bytes = requests.get(img_url, timeout=30).content
image_path = Path("/tmp/example_i2v_input.jpg")
image_path.write_bytes(img_bytes)
display(Image.open(image_path))

data = {
    "prompt": "The scene continues as the robotic arm completes its pick operation and pivots to place the object.",
    "size": "1280x720",
    "num_frames": "189",
    "fps": "24",
    "num_inference_steps": "35",
    "guidance_scale": "6.0",
    "flow_shift": "10.0",
    "seed": "17",
}

print("Sending image-to-video request...")
with image_path.open("rb") as f:
    response = requests.post(
        f"{SERVER_URL}/v1/videos/sync",
        data=data,
        files={"input_reference": (image_path.name, f, "image/jpeg")},
        headers={"Accept": "video/mp4"},
        timeout=300,
    )
response.raise_for_status()

out_path = Path("/tmp/cosmos3_nano_api_i2v.mp4")
out_path.write_bytes(response.content)
print(f"Saved: {out_path}  ({len(response.content) / 1024:.0f} KB)")
Video(str(out_path), width=640)

### 9. Shut Down the Server

In [ ]:
if 'server_proc' in dir() and server_proc.poll() is None:
    server_proc.terminate()
    server_proc.wait(timeout=10)
    print("vLLM-Omni server terminated.")
else:
    print("Server already stopped or was not started in this session.")

---

## Tips & Troubleshooting on AMD ROCm

| Issue | Fix |
|---|---|
| `RuntimeError: HIP error: no kernel image is available` | GPU architecture not in the compiled wheel. Set `HSA_OVERRIDE_GFX_VERSION=11.0.0` (RDNA3) or `9.4.0` (MI300X) |
| OOM during 16B model load | Use `device_map="auto"` to spread across GPUs, or reduce resolution/num_frames |
| BF16 errors on older GPUs | Only CDNA2+ (MI210/MI250/MI300) and RDNA3+ support native BF16; older cards will be slow or error |
| Flash Attention not kicking in | Confirm `FLASH_ATTENTION_TRITON_AMD_ENABLE=TRUE` and that `aotriton` is installed in the container |
| vLLM-Omni server crashes on startup | Try `--enforce-eager` to disable CUDA graph compilation, which can fail on some ROCm configurations |
| Slow first inference on Diffusers path | Normal — PyTorch ROCm compiles Triton kernels on first run; subsequent calls are faster |

## References

- [NVIDIA Cosmos 3 Nano Model Card (HuggingFace)](https://huggingface.co/nvidia/Cosmos3-Nano)
- [Cosmos 3 GitHub](https://github.com/nvidia/cosmos)
- [Cosmos 3 Technical Report](https://research.nvidia.com/labs/cosmos-lab/cosmos3/technical-report.pdf)
- [Diffusers Cosmos3 Documentation](https://huggingface.co/docs/diffusers/main/en/api/pipelines/cosmos3)
- [ROCm Documentation](https://rocm.docs.amd.com)
- [learn-world-model Project](https://github.com/qiwang067/learn-world-model)
